# 17번 보충 실습 · 오지 규정 200페이지 평가

같은 조문과 골든셋으로 노트북의 Chroma와 OVH PostgreSQL·pgvector를 비교합니다. 벡터 검색 개수 k에 따른 적중과 답변 정확도를 확인하고, 실제 답글 API의 작성·검토·저장은 GPT·Claude·DeepSeek로 각각 3회 평가합니다.

[공통 골든셋 원본](data/langsmith_reply_golden_200_v1.json)

In [ ]:
import json

from benchmark.scale.index_200 import ARTIFACTS, BOOK, MANIFEST, load_bundle

corpus = json.loads((ARTIFACTS / "corpus.json").read_text(encoding="utf-8"))
bundle = load_bundle()
print(f"PDF {corpus['pages']}페이지 · 조문 {len(bundle.documents)}개 · 문서 벡터 {bundle.vectors.shape}")
print(f"제1~155조 SHA-256 {corpus['original_155_sha256']}")
print(f"공통 문서 벡터 SHA-256 {bundle.manifest['vector_sha256']}")
print(bundle.documents[0].page_content)

## 재현 명령

실습 폴더의 가상환경에서 실행합니다. PDF를 다시 만들 때는 Windows 맑은 고딕을 사용합니다. 임베딩 단계에는 OpenAI API 사용량이 발생하고, 같은 벡터 번들이 있으면 재사용합니다.

```powershell
uv run --locked --group evaluation python -m benchmark.scale.render_200
uv run --locked --group evaluation python -m benchmark.scale.index_200 embed
uv run --locked --group evaluation python -m benchmark.scale.eval_200 chroma lesson
uv run --locked --group evaluation python -m benchmark.scale.eval_200 chroma api
```

두 저장소 실험은 같은 골든셋·문서 벡터로 비교합니다. OVH 실험 결과는 서버에서 회수한 JSON에서 읽습니다.

In [ ]:
from IPython.display import Markdown, display
from benchmark.scale.eval_200 import GOLDEN, usage_receipt
from benchmark.scale.index_200 import digest

rows = []
for backend in ("chroma", "pgvector"):
    for path in sorted((ARTIFACTS / backend).glob("ohgi-reply-200p-v1-*.json")):
        data = json.loads(path.read_text(encoding="utf-8"))
        config, score = (data[key] for key in ("configuration", "summary"))
        if config["golden_sha256"] != digest(GOLDEN):
            continue
        receipt = usage_receipt(path.with_suffix(".usage.jsonl"), path.with_suffix(".embeddings.jsonl"), config["model"])
        if config["phase"] == "lesson":
            run = f"k={config['k']}"
            outcome = f"검색 {score['retrieval_hit']}/{score['retrieval_eligible']}, 정확도 {score['correct']}/{score['requests']}"
        else:
            run = config["provider"]
            outcome = f"통과 {score['passed']}/{score['requests']}"
        rows.append(f"| {backend} | [{run}]({data['url']}) | {outcome} | ${receipt['measured_usd']} |")
display(Markdown("\n".join([
    "| 저장소 | 실험 | 결과 | 모델 및 질의 임베딩 USD |", "| --- | --- | --- | ---: |", *rows,
])))